# 01 — Preparação e limpeza do INCT SiMAI

**Fonte:** `data.json` fornecido com o enunciado. Esta análise usa apenas essa base.

Objetivo: converter os 71 cenários e 94 bairros em tabela longa, verificar estrutura e coerência, documentar cada correção e salvar os CSVs processados. O cálculo de pessoas afetadas é **estimativa da simulação**, baseada na distribuição uniforme da população entre prédios. O arquivo original fica preservado em `dados/originais`.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
FIG = ROOT / 'resultados' / 'figuras'
FIG.mkdir(parents=True, exist_ok=True)

import pandas as pd
from scripts.preparar_dados import load_inct, clean_inct
raw = load_inct()
print(f'{len(raw)} linhas | {raw.nivel_m.nunique()} níveis | {raw.bairro.nunique()} bairros')
display(raw.head())

6674 linhas | 71 níveis | 94 bairros


/Users/lambari.exe/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,nivel_m,bairro,affected_buildings,total_buildings,percentage,population,affected_population
0,3.0,Aberta dos Morros,0,4696,0,9854,0
1,3.1,Aberta dos Morros,0,4696,0,9854,0
2,3.2,Aberta dos Morros,0,4696,0,9854,0
3,3.3,Aberta dos Morros,0,4696,0,9854,0
4,3.4,Aberta dos Morros,1,4696,1,9854,99


## Auditoria antes das correções

O arquivo usa arredondamento **para cima** em `percentage = ceil(100 × affected_buildings / total_buildings)` e `affected_population = ceil(population × percentage / 100)`. Essa convenção é sistemática e não deve ser confundida com erro. Buscamos divergências contra ela e quedas de prédios atingidos quando o nível aumenta.

In [2]:
import math
check = raw.copy()
check['percentual_esperado'] = (100 * check.affected_buildings / check.total_buildings).apply(math.ceil)
check['populacao_esperada'] = (check.population * check.percentage / 100).apply(math.ceil)
percent_errors = check[check.percentage.ne(check.percentual_esperado)]
pop_errors = check[check.affected_population.ne(check.populacao_esperada)]
decreases = check[check.groupby('bairro').affected_buildings.diff().lt(0)]
print('Percentuais divergentes:', len(percent_errors))
print('População derivada divergente:', len(pop_errors))
print('Quedas entre níveis sucessivos:', len(decreases))
display(percent_errors[['nivel_m','bairro','affected_buildings','total_buildings','percentage','percentual_esperado']])
display(decreases[['nivel_m','bairro','affected_buildings']])

Percentuais divergentes: 2
População derivada divergente: 0
Quedas entre níveis sucessivos: 12


,nivel_m,bairro,affected_buildings,total_buildings,percentage,percentual_esperado
752,7.2,Boa Vista do Sul,686,2450,29,28
753,7.3,Boa Vista do Sul,686,2450,29,28


,nivel_m,bairro,affected_buildings
214,3.1,Arquipélago,4680
219,3.6,Arquipélago,4680
224,4.1,Arquipélago,4674
229,4.6,Arquipélago,4680
234,5.1,Arquipélago,4680
237,5.4,Arquipélago,4680
1274,9.7,Centro Histórico,0
1779,3.4,Espírito Santo,633
1780,3.5,Espírito Santo,25
1793,4.8,Espírito Santo,47


## Correções rastreáveis

As alterações nas contagens são pontuais: série quase constante do Arquipélago, patamar inicial e pico isolado no Espírito Santo, e zeros entre valores positivos no Centro Histórico, Lageado e Rubem Berta. Os campos derivados são então recalculados. A imputação inicial no Espírito Santo é incerta e está identificada no registro de auditoria.

In [3]:
clean, audit, summary = clean_inct()
print(f'{len(clean)} linhas limpas; {len(audit)} alterações de campos')
display(audit.groupby(['bairro','campo']).size().rename('alteracoes').reset_index())
display(audit.head(20))
print('Arquivos salvos em dados/processados/: inct_simai_limpo.csv, auditoria_inct.csv, inct_resumo_por_nivel.csv')

6674 linhas limpas; 41 alterações de campos


,bairro,campo,alteracoes
0,Arquipélago,affected_buildings,7
1,Boa Vista do Sul,affected_population,2
2,Boa Vista do Sul,percentage,2
3,Centro Histórico,affected_buildings,2
4,Centro Histórico,affected_population,2
5,Centro Histórico,percentage,2
6,Espírito Santo,affected_buildings,6
7,Espírito Santo,affected_population,6
8,Espírito Santo,percentage,6
9,Lageado,affected_buildings,1


,bairro,nivel_m,campo,valor_original,valor_corrigido,problema,tratamento
0,Arquipélago,3.0,affected_buildings,4685,4680,Oscilação isolada em série quase constante; ní...,Substituir pela moda observada em 64 dos 71 ní...
1,Arquipélago,3.5,affected_buildings,4685,4680,Oscilação isolada em série quase constante; ní...,Substituir pela moda observada em 64 dos 71 ní...
2,Arquipélago,4.0,affected_buildings,4685,4680,Oscilação isolada em série quase constante; ní...,Substituir pela moda observada em 64 dos 71 ní...
3,Arquipélago,4.1,affected_buildings,4674,4680,Oscilação isolada em série quase constante; ní...,Substituir pela moda observada em 64 dos 71 ní...
4,Arquipélago,4.5,affected_buildings,4685,4680,Oscilação isolada em série quase constante; ní...,Substituir pela moda observada em 64 dos 71 ní...
5,Arquipélago,5.0,affected_buildings,4685,4680,Oscilação isolada em série quase constante; ní...,Substituir pela moda observada em 64 dos 71 ní...
6,Arquipélago,5.3,affected_buildings,4688,4680,Oscilação isolada em série quase constante; ní...,Substituir pela moda observada em 64 dos 71 ní...
18,Boa Vista do Sul,7.2,affected_population,784,757,Campo derivado incompatível com a contagem cor...,Recalcular com arredondamento para cima usado ...
17,Boa Vista do Sul,7.2,percentage,29,28,Campo derivado incompatível com a contagem cor...,Recalcular com arredondamento para cima usado ...
20,Boa Vista do Sul,7.3,affected_population,784,757,Campo derivado incompatível com a contagem cor...,Recalcular com arredondamento para cima usado ...


Arquivos salvos em dados/processados/: inct_simai_limpo.csv, auditoria_inct.csv, inct_resumo_por_nivel.csv


In [4]:
assert not clean.groupby('bairro').affected_buildings.diff().lt(0).any()
assert clean.groupby('nivel_m').bairro.nunique().eq(94).all()
assert clean.affected_buildings.le(clean.total_buildings).all()
print('Validações finais aprovadas.')

Validações finais aprovadas.
